# ddharmon v2 — Split-Aware Harmonization to the NIH CDE Backbone

End-to-end harmonization of biomedical data-dictionary fields to NIH **Common Data Elements (CDEs)**, run
over the public example dictionaries bundled in `data/examples/` using the `ddharmon` `src` modules.

**The v2 pipeline (lean, split-aware):**
1. **Cluster** the cohort fields (BERTopic: UMAP → HDBSCAN).
2. **Retrieve** candidate CDEs per cluster — hybrid lexical (BM25) ⊕ dense (cosine), fused with RRF.
3. **Generate-ideal** — an LLM writes the *ideal* CDE each cluster wants (a coverage anchor).
4. **Split** — partition each cluster into distinct-concept groups on the object/referent axis.
5. **Assign** — per group, the LLM ranks the retrieved CDEs and decides **adopt / refine / novel**.
6. **Route** — adopt/refine → the matched CDE; novel → a GenCDE residual. A retrieval floor downgrades
   geometrically-far "matches" to novel.

Encoder: **BioLORD-2023** (concept↔definition contrastive). Prior art for CDE matching: NIH **CDEMapper**
(we build our own retrieval + assignment). The three LLM stages run through the **Anthropic Batch API**
(schema-enforced, ~50% cheaper). The output is a reviewer-ready **expert-in-the-loop (EITL)** campaign.

> Stages 1–3 (cluster + retrieve) run with **no API key**. The LLM stages (4–6) need `ANTHROPIC_API_KEY`.
> A `MAX_CLUSTERS` cap keeps the demo cheap; remove it for the full corpus.

In [ ]:
# Run from a clone of the ddharmon repo — paths below are relative to the repo root. Installs ddharmon plus
# the extras the pipeline needs (embeddings → sentence-transformers, llm → anthropic Batch API, clustering,
# bertopic) from this repo. No external/dev references.
%pip install -q ".[embeddings,llm,clustering,bertopic]"

In [ ]:
import json
import logging
from pathlib import Path

from ddharmon.ingestion import load_dictionary, preprocess_dictionary
from ddharmon.embedding import SentenceTransformerProvider, embed_dictionary
from ddharmon.clustering import topic_model_dictionaries
from ddharmon.harmonization import (
    prepare_leanb,
    prepare_split,
    prepare_group_assign,
    assemble_leanb,
    write_prompts_jsonl,
    write_records_json,
)
from ddharmon.export.eitl import build_cde_lookup, export_split_eitl_campaign
from ddharmon.llm import submit_and_wait  # Anthropic Batch API (schema-enforced; needs ANTHROPIC_API_KEY)

# Optional: load ANTHROPIC_API_KEY from a local .env if present.
try:
    from dotenv import load_dotenv

    load_dotenv()
except ImportError:
    pass

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub").setLevel(logging.WARNING)
print("Imports OK")

## 1. Load the public cohorts + the NIH CDE backbone

In [ ]:
DATA_DIR = Path("data/examples")  # bundled, publicly-available example dictionaries
CDE_COHORT = "NIH_CDE"

# The NIH CDE catalog is the retrieval/assignment BACKBONE (ships with the full ~22.7k-CDE repo; rebuild or
# subset via `python scripts/flatten_cde_repo.py <input.json> <output.tsv>`). The two public example cohorts
# (All of Us, CLSA) are the fields we harmonize. Bring your own: copy a loader, point it at your CSV/TSV, and
# map your columns onto load_dictionary's named kwargs.
loaders = {
    CDE_COHORT: (DATA_DIR / "all_cdes_flat.tsv", dict(
        variable_name="designation", field_id="tinyId",
        description="definition", question_text="question_text",
        data_type="datatype", value_encoding="permissible_values",
        category="classification", standard_code="concept_codes",
        embed_variable_name=True,
    )),
    "AllOfUs": (DATA_DIR / "all_of_us_surveys.csv", dict(
        variable_name="Item Concept", description="Field Label",
        category="Survey", data_type="Field Type",
        value_encoding="Choices, Calculations, OR Slider Labels",
        question_text="Field Label",
    )),
    "CLSA": (DATA_DIR / "clsa_baseline.csv", dict(
        variable_name="name", short_label="label:en", question_text="question:en",
        description="comment:en", category="table", data_type="valueType",
        units="unit", value_encoding="value_encoding",
    )),
}

cohorts = {}
for name, (path, kwargs) in loaders.items():
    if not path.exists():
        print(f"{name}: SKIPPED (not found: {path})")
        continue
    cohorts[name] = preprocess_dictionary(load_dictionary(path, cohort_name=name, **kwargs))
    print(f"{name}: {cohorts[name].field_count} fields")

COHORTS = [n for n in cohorts if n != CDE_COHORT]  # the cohorts we cluster + harmonize
print(f"\nHarmonizing {COHORTS} against the {CDE_COHORT} backbone")

## 2. Embed (BioLORD-2023)

`SentenceTransformerProvider()` defaults to **BioLORD-2023** (768d). Embeddings are cached in
`.ddharmon/embeddings.db` keyed by `(model, content, vector_type)`, so re-runs are free.

> First run embeds the ~22.7k-CDE backbone — a few minutes on CPU, then cached.

In [ ]:
provider = SentenceTransformerProvider()  # BioLORD-2023 by default
embedded = {name: embed_dictionary(dd, provider=provider) for name, dd in cohorts.items()}
embedded_list = list(embedded.values())            # full set incl. CDE backbone (for retrieval)
cohort_embedded = [embedded[n] for n in COHORTS]   # the fields we cluster (CDEs are the backbone, not clustered)
for name, ed in embedded.items():
    print(f"  {name}: {len(ed.embeddings)} vectors")

## 3. Cluster → retrieve candidates → build the generate-ideal prompts  ($0 — no API key)

Cluster the cohort fields, retrieve a hybrid (BM25 ⊕ dense) top-k of candidate CDEs per cluster, and prepare
the stage-1 *generate-ideal* prompts. `MAX_CLUSTERS` keeps the demo cheap by harmonizing only the largest
(most heterogeneous) clusters — set it to `None` for the full corpus.

In [ ]:
MAX_CLUSTERS = 25  # demo cap (largest clusters first) → bounds LLM cost. Set None for the full corpus.

tm = topic_model_dictionaries(cohort_embedded, min_cluster_size=15)
print(f"{len(tm.clusters)} clusters over {len(tm.field_refs)} cohort fields")

ideal_prompts = prepare_leanb(
    tm.clusters, embedded_list, tm.embeddings, tm.field_refs, cde_cohort=CDE_COHORT, top_k=20
)
ideal_prompts.sort(key=lambda r: len(r.context["members"]), reverse=True)
if MAX_CLUSTERS:
    ideal_prompts = ideal_prompts[:MAX_CLUSTERS]
print(f"{len(ideal_prompts)} clusters queued for harmonization (MAX_CLUSTERS={MAX_CLUSTERS})")

# Peek at one cluster: its member fields + the CDE candidates retrieved for it ($0, no LLM yet).
ctx = ideal_prompts[0].context
print("\nexample cluster members:", [m["variable_name"] for m in ctx["members"][:6]], "…")
print("retrieved CDE candidates:", [c["designation"] for c in ctx["candidates"][:5]])

## 4. The three LLM stages via the Batch API (schema-enforced) — needs `ANTHROPIC_API_KEY`

Each stage carries a JSON **schema** that the Batch API appends to the prompt, so the model returns exactly
the fields the next stage parses (split → `member_ids`; assign → `cde_id` + `rationale`). Inline single-shot
completion does **not** show the schema and silently drops these — always run these stages through the batch.
Responses are cached per stage under `harmonization_artifacts/`, so a re-run is free.

In [ ]:
WORK = Path("harmonization_artifacts")
WORK.mkdir(exist_ok=True)
MODEL = "claude-sonnet-4-6"


def run_stage(records, tag):
    """Run one LLM stage via the Batch API (schema-enforced per record) → {id: response}."""
    prompts_path, responses_path = WORK / f"prompts_{tag}.jsonl", WORK / f"responses_{tag}.jsonl"
    write_prompts_jsonl(records, prompts_path)
    submit_and_wait(prompts_path, responses_path, model=MODEL, max_tokens=1024)  # needs ANTHROPIC_API_KEY
    responses = {}
    with open(responses_path) as f:
        for line in f:
            rec = json.loads(line)
            responses[rec["id"]] = rec["response"]
    return responses


# stage 1 — generate ideal CDE per cluster
gen = run_stage(ideal_prompts, "generate_ideal")
# stage 2 — split each cluster into distinct-concept groups
split_prompts = prepare_split(ideal_prompts, gen, model_tag=MODEL)
sresp = run_stage(split_prompts, "split")
# stage 3 — per concept-group, re-retrieve + assign adopt/refine/novel
group_prompts = prepare_group_assign(
    split_prompts, sresp, embedded_list, tm.embeddings, tm.field_refs, cde_cohort=CDE_COHORT, top_k=20, model_tag=MODEL
)
aresp = run_stage(group_prompts, "assign")

# route: adopt/refine → matched CDE, novel → GenCDE residual; floor downgrades far matches
result = assemble_leanb(group_prompts, aresp, retrieval_floor=0.30)
print(f"{len(result.records)} routed records")

## 5. Routed records → verdict buckets + a reviewer-ready EITL campaign

In [ ]:
buckets = result.buckets()
print("verdict buckets:", {k: len(v) for k, v in buckets.items()})

# machine-readable records + the expert-in-the-loop review campaign (contract-clean CSVs)
write_records_json(result, WORK / "records.json")
cde_lookup = build_cde_lookup(cohorts[CDE_COHORT])
source_dicts = {n: cohorts[n] for n in COHORTS}
counts = export_split_eitl_campaign(
    result, source_dicts, cde_lookup, out_dir=WORK / "eitl", stem="ddharmon",
    embedded={n: embedded[n] for n in COHORTS},
)
print("EITL campaign rows:", counts)
print("written to", WORK / "eitl")

## 6. Inspect the verdicts

In [ ]:
import pandas as pd

df = pd.DataFrame(
    [
        {
            "cluster": r.cluster_id,
            "group": r.group_id,
            "concept": r.concept,
            "verdict": r.verdict,
            "route": r.route,
            "cde": r.cde_id,
            "cde_id": r.cde_external_id,
            "chosen_cos": r.chosen_cos,
            "cross_cohort": r.cross_cohort,
            "n_fields": r.n_members,
            "cohorts": ";".join(r.cohorts),
        }
        for r in result.records
    ]
)
print(df["verdict"].value_counts())
df.sort_values(["verdict", "chosen_cos"], ascending=[True, True]).head(30)